# Coding Project 2: Credit Analytics

**Names of all group members:**


---

All code below is only suggestive and you may as well use different approaches.
Make sure that all requested quantities are printed or shown in tables/plots in the executed notebook.
A concise *Project Report* is required at the end of the notebook (see the last cells).

In [17]:
# Exercise 1.
import numpy as np
np.random.seed(0)  # for reproducibility

# simulate explanatory variables x (the age and monthly income are assumed here to follow a continuous uniform distribution)
m = 20000
n = 10000

# age in [18, 80], monthly income in [1, 15] (in CHF 1000), both unirmly distributed
x_1 = np.random.uniform(18, 80, size=(n+m, 1))  # age
x_2 = np.random.uniform(1, 15, size=(n+m, 1))  # monthly income

# 0: salaried, 1 = self-employed with probability of being self-employed = 0.1
x_3 = np.random.binomial(1, 0.1, size=(n+m, 1))  # employment status

# vector in R^3 of explanatory variables using only m training data points (the first m rows of x_1, x_2, and x_3)
x_ = np.hstack((x_1, x_2, x_3))
x_train = x_[:m, :]
x_test = x_[m:, :]
# a) calculate and print the empirical means and standard deviations over the training data (i = 1, ..., m)

print("Empirical means and standard deviations over the training data:")
print(" ")

for i, x in enumerate([x_1, x_2, x_3], start=1):
    print(f"Feature {i}: Mean = {np.mean(x[:m]):.2f}, Std = {np.std(x[:m]):.2f}")



# b) Can you come up with a few (2 or 3) additional features that may be relevant?
# (you don't have to implement those of course, just write down your answer in text)
#
print(" ")
print("Additional features that may be relevant:")
print("1. Family size: The number of dependents or family members could influence financial decisions and risk profiles.")
print("2. Net worth score: A person's net worth is a key indicator of their financial stability and ability to repay debt.")
print("3. Loan amount: The amount of the loan requested can impact the risk and terms of the loan.")


Empirical means and standard deviations over the training data:
 
Feature 1: Mean = 48.74, Std = 18.01
Feature 2: Mean = 7.99, Std = 4.03
Feature 3: Mean = 0.10, Std = 0.30
 
Additional features that may be relevant:
1. Family size: The number of dependents or family members could influence financial decisions and risk profiles.
2. Net worth score: A person's net worth is a key indicator of their financial stability and ability to repay debt.
3. Loan amount: The amount of the loan requested can impact the risk and terms of the loan.


In [20]:
# Exercise 2.
# Building the datasets:


# logistic sigmoid function
sigmoid = lambda x: 1. / (1. + np.exp(-x))

# random variables uniforml distirbuted
xi = np.random.uniform(0, 1, size=x_1.shape)

# probability that borrower i is "good" in 1,2
p_1 = sigmoid(13.3 - 0.33 *x_1 + 3.5 * x_2 - 3 * x_3)
p_2 = sigmoid(5 - 10 * ((x_1 < 25).astype(int) + (x_1 > 75).astype(int)) + 1.1 * x_2- x_3)

# build first and second datasets (with different underlying probabilities) 
y_1 = (xi <= p_1).astype(int)
y_2 = (xi <= p_2).astype(int)

y_1_train = y_1[:m]
y_2_train = y_2[:m]

y_1_test = y_1[m:]
y_2_test = y_2[m:]


In [23]:
# Exercise 2. a)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
# "model = LogisticRegression().fit(X_data, Y_data)" fits a model
# "pred_X = model.predict_proba(X)" evaluates the model
# (note that it outputs both P(Y=0|X) and P(Y=1|X))
# "log_loss(Y, pred_X)" evaluates the negative conditional log likelihood (also called cross-entropy loss)

# Fit the models on both datasets
model_1 = LogisticRegression().fit(x_train, y_1_train)  # Use only the training data for y_1
model_2 = LogisticRegression().fit(x_train, y_2_train)  # Use only the training data for y_2

pred_X_1= model_1.predict_proba(x_)
pred_X_2 = model_2.predict_proba(x_)

cross_entropy_loss_1_train = log_loss(y_1_train, pred_X_1[:m])  # Use only the training data for y_1
cross_entropy_loss_2_train = log_loss(y_2_train, pred_X_2[:m])

cross_entropy_loss_1_test = log_loss(y_1_test, pred_X_1[m:])  # Use only the test data for y_1
cross_entropy_loss_2_test = log_loss(y_2_test, pred_X_2[m:])


# Calculate and print the cross-entropy loss on both datasets for train and test

print("Cross-entropy loss on the training data:")
print(f"Dataset 1: {cross_entropy_loss_1_train:.4f}")
print(f"Dataset 2: {cross_entropy_loss_2_train:.4f}")
print(" ")
print("Cross-entropy loss on the test data:")
print(f"Dataset 1: {cross_entropy_loss_1_test:.4f}")
print(f"Dataset 2: {cross_entropy_loss_2_test:.4f}")


Cross-entropy loss on the training data:
Dataset 1: 0.0316
Dataset 2: 0.1584
 
Cross-entropy loss on the test data:
Dataset 1: 0.0340
Dataset 2: 0.1565


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/utils/validation.py:1368: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [ ]:
# Exercise 2. b)
# Calculate normalized data
# (as in the assignment: divide by the empirical standard deviations of the TRAINING data,
#  for both training and test data)

x_train_mean = np.mean(x_train, axis=0)
x_train_std = np.std(x_train, axis=0)

print("Training data mean:", x_train_mean)
print("Training data standard deviation:", x_train_std)
print("We normalized by computing: (x - mean) / std for each feature.")

x_train_normalized = (x_train - x_train_mean) / x_train_std


In [ ]:
# Exercise 2.b) (i) and (ii)
from sklearn.svm import SVC
# "model = SVC(kernel='rbf', gamma=GAMMA, C=C, probability=True)" creates
# a model with kernel exp(-GAMMA \|x-x'\|_2^2) and regularization parameter C 
# (note the relation between C and the parameter lambda from the lecture).
# "probability=True" enables the option "model.predict_proba(X)" to predict probabilities from the regression function \hat{f}^{svm}.
# "model.fit(X, Y)" optimizes the model parameters (using hinge loss)

# Fit the models for both datasets (this can take up to 60 seconds with SVC)
# Note: "model.fit(...)" modifies the model object in place, so use a separate model object
# for each dataset if you want to keep both fitted models.



In [ ]:
# Exercise 2.b (iii)
# "model.predict_proba(X)" predicts probabilities from features (note that it outputs both P(Y=0|X) and P(Y=1|X))

# Calculate and print the cross-entropy loss on both normalized datasets for train and test


In [ ]:
# Exercise 2.b (iv)
#Would the results change using standardized data instead of normalized data? No need to run any code here, only explain your reasoning.

In [ ]:
# Exercise 2.c
# Plot the FDR/TPR-curves with the TPR on the horizontal axis and the FDR on the vertical axis,
# and compute the AUC of each plotted curve on the test data.
import matplotlib.pyplot as plt
# Thresholds c: you may use all distinct predicted probabilities (this gives the exact curve), or a grid of
# values in [0, 1]. A uniform grid such as np.linspace(0, 1, 100) can be too coarse when many predicted
# probabilities are very close to 1. Check that your curve covers TPR values from (close to) 0 to 1.

# To approximately calculate the AUC, it is fine to simply use Riemann sums.
# This means, if you have N (a_i, b_i) pairs for the curves, ordered such that a_1 <= a_2 <= ... <= a_N,
# then you may simply use the sum
# sum_{i=1}^{N-1} (b_i + b_{i+1})/2 * (a_{i+1}-a_i)
# as the approximation of the integral (or AUC)


# first data set & logistic regression:
# (the code should be reusable for all cases, only exchanging datasets and predicted probabilities depending on the model)

# Calculate positives (only depending on the dataset)

# Calculate true positives for all threshold values

# Calculate false positives for all threshold values

# Calculate FDR and TPR rate (points on the FDR/TPR curve) and the AUC


# second data set & logistic regression:


# first data set and SVM:


# second data set and SVM:


# Print the AUC values.
# Briefly explain (in a Markdown cell or comments) how you construct the curves, interpret what
# the AUC means in this setting, and compare the two models on both data sets.

In [ ]:
# Exercise 3.

# Set model parameters and define matrix D
# (D has n x 50000 = 5*10^8 entries: a float64 array of uniforms of this size needs about 4 GB of memory.
#  If your machine runs out of memory, consider generating D in blocks of columns and/or storing it as booleans.)


# Strategy (i):
# Define Portfolio and possible outcomes for this portfolio using matrix D


# Plot histogram of profits and losses


# Calculate expected profit and losses and 95%-VaR


# Strategy (ii):
# Define Portfolio and possible outcomes using the matrix D and the predicted repayment probabilities from the logistic regression model


# Plot histogram of profits and losses


# Calculate expected profit and losses and 95%-VaR


# Strategy (iii):
# Define Portfolio and possible outcomes using the matrix D and the predicted repayment probabilities from the SVM model
# (evaluated on the normalized test features x~_i, see the assignment)


# Plot histogram of profits & losses (which is simply the performance of each strategy in scenario k, i.e. a profit if positive or a loss if negative)


# Calculate and print the expected profit & loss and 95%-VaR for each strategy (in CHF)


# c) Compare the three strategies and briefly interpret your results (Markdown cell or comments).

### Project Report

At the end of the notebook, provide a concise synthesis of your analysis. Do not simply repeat every answer above.

#### Approach
Briefly summarize how you generated the data, which models you fitted, and how you evaluated them and the lending strategies.

#### Model selection
Explain which classifier you would rely on for the lending decision and why. (The model hyperparameters are fixed by the assignment; no hyperparameter tuning is required.)

#### Key results
Summarize the most important quantitative results.

#### Interpretation
Explain what the results imply about the classification methods and the lending problem.

#### Robustness
Discuss how strongly your conclusions could depend on random elements of the analysis (e.g. simulation noise in the data and in the market scenarios). No additional experiments are required.

#### Limitations
State the main limitations of the analysis, the models, and the modelling assumptions.